<div align="center">
  <a href="https://roboflow.com" target="_blank">
    <img
      width="100%"
      src="https://media.roboflow.com/notebooks/template/bannerformats-dark.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672949716562"
    >
  </a>
</div>

# Fine-Tune YOLO26 on Pose Estimation Dataset

---

[![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://roboflow.com)
[![YouTube](https://badges.aleen42.com/src/youtube.svg)](https://youtube.com/roboflow)
[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/ultralytics/ultralytics)

YOLO26 introduces **NMS-free end-to-end inference** and the **MuSGD optimizer**, making it faster and more accurate than previous YOLO versions. This notebook shows you how to fine-tune YOLO26 on a custom pose estimation dataset using [Roboflow](https://roboflow.com) for dataset management.

**What you'll learn:**
- How to prepare a pose estimation dataset with Roboflow
- How to fine-tune YOLO26 on a custom keypoint dataset
- How to run inference and visualize keypoints
- How to export the model for deployment

**Before you start**, make sure you have access to a GPU. You can use a **free T4 GPU** by navigating to `Runtime` → `Change runtime type` → `T4 GPU`.

## Step 1: Install Dependencies

In [ ]:
import os
HOME = os.getcwd()
print("HOME:", HOME)

In [ ]:
%pip install ultralytics supervision roboflow -q

import ultralytics
ultralytics.checks()

## Step 2: Load Pretrained YOLO26 Pose Model

In [ ]:
from ultralytics import YOLO

# Load YOLO26 nano pose model pretrained on COCO
model = YOLO("yolo26n-pose.pt")
print("Model loaded successfully!")

## Step 3: Download Dataset from Roboflow

We'll use a publicly available pose estimation dataset from Roboflow Universe. You can replace this with your own dataset.

**To use your own dataset:**
1. Go to [Roboflow Universe](https://universe.roboflow.com)
2. Find or upload your pose estimation dataset
3. Export in **YOLOv8 Pose** format
4. Replace the snippet below with your own export code

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Replace with your Roboflow API key
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov8-pose")

print(f"Dataset downloaded to: {dataset.location}")

## Step 4: Fine-Tune YOLO26 on Pose Estimation Dataset

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n-pose.pt")

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project="pose_training",
    name="yolo26n_pose",
    exist_ok=True,
    patience=20,
    save=True,
    plots=True,
)

print(f"\nTraining complete!")
print(f"Best model saved to: {results.save_dir}")

## Step 5: Evaluate the Model

In [ ]:
from ultralytics import YOLO

# Load best trained model
model = YOLO("pose_training/yolo26n_pose/weights/best.pt")

# Run validation
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    device=0,
)

print(f"Pose mAP50:   {metrics.pose.map50:.4f}")
print(f"Pose mAP50-95: {metrics.pose.map:.4f}")
print(f"Box mAP50:    {metrics.box.map50:.4f}")

## Step 6: Run Inference and Visualize Keypoints

In [ ]:
import glob
import random
from IPython.display import display, Image as IPImage
from ultralytics import YOLO

model = YOLO("pose_training/yolo26n_pose/weights/best.pt")

# Pick a random validation image
val_images = glob.glob(f"{dataset.location}/valid/images/*.jpg")
test_image = random.choice(val_images)

results = model.predict(
    source=test_image,
    conf=0.35,
    save=True,
    project="pose_inference",
    name="yolo26_pose_results",
    exist_ok=True,
)

# Display result
output_image = glob.glob("pose_inference/yolo26_pose_results/*.jpg")[0]
display(IPImage(output_image, width=800))

# Print keypoint details
for r in results:
    if r.keypoints is not None:
        print(f"Detected {len(r.keypoints)} person(s)")
        print(f"Keypoints shape: {r.keypoints.xy.shape}")

## Step 7: Visualize with Supervision

In [ ]:
import supervision as sv
import cv2
from ultralytics import YOLO
from IPython.display import display, Image as IPImage
import tempfile, os

model = YOLO("pose_training/yolo26n_pose/weights/best.pt")

image = cv2.imread(test_image)
results = model(image, conf=0.35)[0]

# Extract detections and keypoints
keypoints = sv.KeyPoints.from_ultralytics(results)
detections = sv.Detections.from_ultralytics(results)

# Annotate
box_annotator = sv.BoxAnnotator()
edge_annotator = sv.EdgeAnnotator(color=sv.Color.GREEN, thickness=2)
vertex_annotator = sv.VertexAnnotator(color=sv.Color.RED, radius=4)

annotated = box_annotator.annotate(image.copy(), detections)
annotated = edge_annotator.annotate(annotated, keypoints)
annotated = vertex_annotator.annotate(annotated, keypoints)

# Save and display
output_path = "pose_supervision_output.jpg"
cv2.imwrite(output_path, annotated)
display(IPImage(output_path, width=800))

## Step 8: Export for Deployment

In [ ]:
from ultralytics import YOLO

model = YOLO("pose_training/yolo26n_pose/weights/best.pt")

# Export to ONNX
model.export(format="onnx", imgsz=640, simplify=True)
print("Exported to ONNX")

# Export to TFLite (mobile devices)
# model.export(format="tflite", imgsz=640, int8=True)

# Export to TensorRT (NVIDIA GPUs)
# model.export(format="engine", imgsz=640, half=True)

## What's Next?

- 📖 [YOLO26 Docs](https://docs.ultralytics.com/models/yolo26)
- 🔍 [Roboflow Universe — Pose Datasets](https://universe.roboflow.com/search?q=pose+estimation)
- 💜 [Supervision Docs](https://supervision.roboflow.com)
- 🐛 [Report Issues](https://github.com/roboflow/notebooks/issues)

<div align="center">
  <div>
    <a href="https://youtube.com/roboflow">
        <img
          src="https://media.roboflow.com/notebooks/template/icons/purple/youtube.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672949748453"
          height="110"
        >
    </a>
    <a href="https://roboflow.com">
        <img
          src="https://media.roboflow.com/notebooks/template/icons/purple/roboflow-app.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672949748453"
          height="110"
        >
    </a>
    <a href="https://www.linkedin.com/company/roboflow-ai/">
        <img
          src="https://media.roboflow.com/notebooks/template/icons/purple/linkedin.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672949748453"
          height="110"
        >
    </a>
    <a href="https://roboflow.com/twitter">
        <img
          src="https://media.roboflow.com/notebooks/template/icons/purple/twitter.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672949748453"
          height="110"
        >
    </a>
    <a href="https://discord.gg/roboflow">
        <img
          src="https://media.roboflow.com/notebooks/template/icons/purple/discord.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672949748453"
          height="110"
        >
    </a>
  </div>
</div>